# Raw Ingestion — NASA Exoplanet Archive
**Pipeline stage:** `RAW`

## Objetivo científico
Construir un pipeline de análisis de población estelar usando el catálogo PSCompPars de la NASA.
Estudiamos las propiedades de las estrellas anfitrionas (temperatura, luminosidad, masa, tipo espectral)
para construir un Diagrama de Hertzsprung-Russell y analizar cómo las características estelares
se relacionan con la arquitectura de sus sistemas planetarios.

### Columnas seleccionadas:
| Grupo | Columnas |
|---|---|
| **Identificadores** | `pl_name`, `hostname` |
| **Sistema** | `sy_snum`, `sy_pnum`, `sy_dist` |
| **Estrella (HR)** | `st_teff`, `st_rad`, `st_mass`, `st_lum`, `st_logg`, `st_met`, `st_spectype` |
| **Planeta** | `pl_orbper`, `pl_rade`, `pl_bmasse`, `pl_eqt` |
| **Descubrimiento** | `disc_year`, `discoverymethod` |

In [2]:
import os, hashlib, json, requests
import pandas as pd
from datetime import datetime, timezone

os.makedirs("../data/raw", exist_ok=True)
print(os.getcwd())

c:\Users\nancy\Documents\Universidad\Fisica Computacional I\Proyecto_I\notebooks


## Descarga desde NASA TAP API

In [3]:
TAP_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

COLUMNS = [
    "pl_name", "hostname",
    "sy_snum", "sy_pnum", "sy_dist",
    "st_spectype", "st_teff", "st_rad", "st_mass",
    "st_lum", "st_logg", "st_met",
    "pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt",
    "disc_year", "discoverymethod",
]

query = f"select {','.join(COLUMNS)} from pscomppars"
params = {"query": query, "format": "csv"}

print("Descargando PSCompPars...")
resp = requests.get(TAP_URL, params=params, timeout=120)
resp.raise_for_status()

TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RAW_PATH  = f"../data/raw/pscomppars_stellar_{TIMESTAMP}.csv"

with open(RAW_PATH, "wb") as f:
    f.write(resp.content)

print(f"Guardado: {RAW_PATH}")
print(f"Tamaño  : {os.path.getsize(RAW_PATH):,} bytes")

Descargando PSCompPars...
Guardado: ../data/raw/pscomppars_stellar_20260406T020226Z.csv
Tamaño  : 957,455 bytes


## Hash SHA-256 — integridad del dato crudo

In [4]:
SHA256 = hashlib.sha256(resp.content).hexdigest()
MD5    = hashlib.md5(resp.content).hexdigest()

df_raw = pd.read_csv(RAW_PATH)
N_ROWS, N_COLS = df_raw.shape

print(f"SHA-256 : {SHA256}")
print(f"MD5     : {MD5}")
print(f"Filas   : {N_ROWS:,}")
print(f"Columnas: {N_COLS}")

SHA-256 : 768f7b3ee02e6846688692a86f558c1076e2b8169530f0e04351d0b4399a00ec
MD5     : 8aebce77cdfc8e7efc4d9a96ff09b4f6
Filas   : 6,153
Columnas: 18


## Guardar metadata de ingestión

In [5]:
metadata = {
    "source"                : "NASA Exoplanet Archive — PSCompPars",
    "scientific_focus"      : "Stellar population & HR-diagram analysis",
    "table"                 : "pscomppars",
    "columns"               : COLUMNS,
    "tap_url"               : TAP_URL,
    "download_timestamp_utc": TIMESTAMP,
    "raw_file"              : RAW_PATH,
    "sha256"                : SHA256,
    "md5"                   : MD5,
    "n_rows"                : int(N_ROWS),
    "n_cols"                : int(N_COLS),
    "pipeline_version"      : "1.0.0",
}

META_PATH = f"../data/raw/metadata_{TIMESTAMP}.json"
with open(META_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

# apuntador al más reciente (para notebooks siguientes)
with open("../data/raw/latest.json", "w") as f:
    json.dump({"latest_raw": RAW_PATH, "latest_meta": META_PATH}, f, indent=2)

print("Metadata guardada")
print(json.dumps(metadata, indent=2))

Metadata guardada
{
  "source": "NASA Exoplanet Archive \u2014 PSCompPars",
  "scientific_focus": "Stellar population & HR-diagram analysis",
  "table": "pscomppars",
  "columns": [
    "pl_name",
    "hostname",
    "sy_snum",
    "sy_pnum",
    "sy_dist",
    "st_spectype",
    "st_teff",
    "st_rad",
    "st_mass",
    "st_lum",
    "st_logg",
    "st_met",
    "pl_orbper",
    "pl_rade",
    "pl_bmasse",
    "pl_eqt",
    "disc_year",
    "discoverymethod"
  ],
  "tap_url": "https://exoplanetarchive.ipac.caltech.edu/TAP/sync",
  "download_timestamp_utc": "20260406T020226Z",
  "raw_file": "../data/raw/pscomppars_stellar_20260406T020226Z.csv",
  "sha256": "768f7b3ee02e6846688692a86f558c1076e2b8169530f0e04351d0b4399a00ec",
  "md5": "8aebce77cdfc8e7efc4d9a96ff09b4f6",
  "n_rows": 6153,
  "n_cols": 18,
  "pipeline_version": "1.0.0"
}


## Checks de sanidad (evidencia)

In [6]:
print("=== Shape ===")
print(f"{df_raw.shape[0]:,} filas  x  {df_raw.shape[1]} columnas")

print("\n=== Nulos por columna ===")
nulls = df_raw.isnull().sum().sort_values(ascending=False)
print(nulls[nulls > 0].to_string())

print(f"\n=== Estrellas únicas (hostname) : {df_raw['hostname'].nunique():,}")
print(f"=== Planetas únicos  (pl_name)  : {df_raw['pl_name'].nunique():,}")

print("\n=== Métodos de descubrimiento ===")
print(df_raw["discoverymethod"].value_counts().head(8).to_string())

print("\n=== Rango de temperaturas estelares (st_teff) ===")
print(df_raw["st_teff"].describe())

df_raw.head(3)

=== Shape ===
6,153 filas  x  18 columnas

=== Nulos por columna ===
st_spectype    3845
pl_eqt         1563
st_met          550
pl_orbper       334
st_logg         318
st_rad          314
st_lum          308
st_teff         290
pl_rade          50
pl_bmasse        31
sy_dist          27
st_mass           8
disc_year         1

=== Estrellas únicas (hostname) : 4,585
=== Planetas únicos  (pl_name)  : 6,153

=== Métodos de descubrimiento ===
discoverymethod
Transit                          4520
Radial Velocity                  1182
Microlensing                      275
Imaging                            94
Transit Timing Variations          39
Eclipse Timing Variations          17
Orbital Brightness Modulation       9
Pulsar Timing                       8

=== Rango de temperaturas estelares (st_teff) ===
count     5863.000000
mean      5397.502857
std       1747.337620
min        415.000000
25%       4903.845000
50%       5545.000000
75%       5900.000000
max      57000.000000
Name: st

,pl_name,hostname,sy_snum,sy_pnum,sy_dist,st_spectype,st_teff,st_rad,st_mass,st_lum,st_logg,st_met,pl_orbper,pl_rade,pl_bmasse,pl_eqt,disc_year,discoverymethod
0,Kepler-1167 b,Kepler-1167,1,1,820.905,NaN,4971.0,0.750,0.790,-0.53589,4.600,-0.05,1.003934,1.710000,3.570,1419.0,2016.0,Transit
1,Kepler-1740 b,Kepler-1740,1,1,1061.770,NaN,5705.0,0.905,0.943,-0.07942,4.499,-0.06,8.172400,3.323214,11.000,858.0,2021.0,Transit
2,Kepler-1581 b,Kepler-1581,1,1,493.175,NaN,6022.0,1.230,1.120,0.39085,4.310,0.07,6.283855,0.800000,0.437,1108.0,2016.0,Transit


## Checkpoint RAW
| Check | Estado |
|---|---|
| Descarga exitosa | ✓ |
| Hash SHA-256 registrado | ✓ |
| Metadata JSON guardada | ✓ |
| CSV con timestamp | ✓ |
| Checks de sanidad ejecutados | ✓ |